# Phase 2 — Getting Data Out of Arrays and Putting Them Together

Phase 1 was about *making* arrays. This phase is about the everyday work: **pulling out the
bits you want** and **joining arrays together**.

If you've ever used a spreadsheet, you already understand the goal:
- highlight a block of cells → that's **slicing**
- "show me only the rows where sales > 1000" → that's **filtering**
- click the header to sort by a column → that's **argsort**
- paste one table under another → that's **stacking**

### What this notebook covers
1. Cutting up a simple row of numbers (slicing)
2. Picking from rows and columns (2-D indexing)
3. Filtering — "only give me the ones where..."
4. Making decisions without writing `if` statements
5. Sorting, including sorting a whole table by one column
6. Joining arrays together
7. Adding and removing items (and why it's slower than you'd think)
8. Cutting an array into pieces

In [1]:
# ==========================================
# SETUP
# ==========================================
import numpy as np

## 1. Cutting up a row of numbers

The pattern is `arr[start:stop:step]` — identical to Python lists, so if you know one you
know the other.

Read it as: **"start here, stop just before there, moving this many at a time."**

Three things to burn in:
- `start` **is included**, `stop` **is not**. `arr[2:4]` gives you items 2 and 3.
- Leave a part out and NumPy assumes the obvious: beginning, end, one at a time.
- Negative numbers count backwards from the end. `-1` is the last item, `-2` the second-last.

In [2]:
# ==========================================
# 1. SLICING A 1-D ARRAY
# ==========================================
arr = np.array([10, 20, 30, 40, 50])
#   position:     0   1   2   3   4      ← counting starts at 0, not 1

print("the array      :", arr)
print("arr[2:4]       :", arr[2:4],   "→ items 2 and 3 (it stops BEFORE 4)")
print("arr[:3]        :", arr[:3],    "→ no start, so from the beginning: first three")
print("arr[3:]        :", arr[3:],    "→ no stop, so from item 3 to the end")
print("arr[1:5:2]     :", arr[1:5:2], "→ from 1 up to 5, skipping every other one")
print("arr[-1]        :", arr[-1],    "→ the last item")
print("arr[-2:]       :", arr[-2:],   "→ the last two")
print("arr[::-1]      :", arr[::-1],  "→ a step of -1 walks backwards = reversed")

the array      : [10 20 30 40 50]
arr[2:4]       : [30 40] → items 2 and 3 (it stops BEFORE 4)
arr[:3]        : [10 20 30] → no start, so from the beginning: first three
arr[3:]        : [40 50] → no stop, so from item 3 to the end
arr[1:5:2]     : [20 40] → from 1 up to 5, skipping every other one
arr[-1]        : 50 → the last item
arr[-2:]       : [40 50] → the last two
arr[::-1]      : [50 40 30 20 10] → a step of -1 walks backwards = reversed


### A trap: `arr[4:0:-1]` doesn't reach the first item

When you walk backwards, `stop` is *still* excluded — so position 0 gets skipped and you
silently lose your first value.

Just use `arr[::-1]` to reverse. It's shorter and it can't go wrong.

In [3]:
print("arr[4:0:-1] :", arr[4:0:-1],  "← where did 10 go? 'stop at 0' means 'stop before 0'")
print("arr[4::-1]  :", arr[4::-1],   "← no stop given, so it runs all the way back")
print("arr[::-1]   :", arr[::-1],    "← same result, and much easier to read")

arr[4:0:-1] : [50 40 30 20] ← where did 10 go? 'stop at 0' means 'stop before 0'
arr[4::-1]  : [50 40 30 20 10] ← no stop given, so it runs all the way back
arr[::-1]   : [50 40 30 20 10] ← same result, and much easier to read


> **Remember from Phase 1:** a slice is a *window*, not a copy. Assigning into a slice
> changes the original array. That's genuinely useful when you mean it — it lets you update
> a chunk of an array in one line.

In [4]:
arr = np.array([10, 20, 30, 40, 50])
arr[1:4] = 0                          # overwrite three items at once, no loop needed
print("after arr[1:4] = 0 :", arr)

after arr[1:4] = 0 : [10  0  0  0 50]


## 2. Picking from rows and columns

For a 2-D array (a table), give **one index per direction, separated by a comma**:

```
arr[row, column]
```

Each side takes either a single number ("just this one") or a slice ("this range").
The magic word is `:` on its own, meaning **"all of them"**:

- `arr[0, :]` → row 0, all columns → **one whole row**
- `arr[:, 1]` → all rows, column 1 → **one whole column**

In [5]:
# ==========================================
# 2. INDEXING A 2-D ARRAY (A TABLE)
# ==========================================
arr2d = np.array([[1, 2, 3],      # row 0
                  [4, 5, 6],      # row 1
                  [7, 8, 9]])     # row 2
#                  col0 col1 col2

print("the table:\n", arr2d)

print("\narr2d[1, 2]     :", arr2d[1, 2],  "→ row 1, column 2 → a single number")
print("arr2d[0, :]     :", arr2d[0, :],  "→ row 0, every column → the whole first row")
print("arr2d[:, 1]     :", arr2d[:, 1],  "→ every row, column 1 → the whole middle column")
print("\narr2d[0:2, 1:3] :\n", arr2d[0:2, 1:3], "\n→ rows 0-1 crossed with cols 1-2: a block")
print("\narr2d[::2, ::2] :\n", arr2d[::2, ::2], "\n→ every other row AND column: the corners")
print("\narr2d[-1, -1]   :", arr2d[-1, -1], "→ last row, last column: bottom-right")

the table:
 [[1 2 3]
 [4 5 6]
 [7 8 9]]

arr2d[1, 2]     : 6 → row 1, column 2 → a single number
arr2d[0, :]     : [1 2 3] → row 0, every column → the whole first row
arr2d[:, 1]     : [2 5 8] → every row, column 1 → the whole middle column

arr2d[0:2, 1:3] :
 [[2 3]
 [5 6]] 
→ rows 0-1 crossed with cols 1-2: a block

arr2d[::2, ::2] :
 [[1 3]
 [7 9]] 
→ every other row AND column: the corners

arr2d[-1, -1]   : 9 → last row, last column: bottom-right


### Write `arr[i, j]`, not `arr[i][j]`

Both give the same answer, but they do different amounts of work.

`arr[i][j]` is two separate steps: first pull out row `i` as its own array, then look inside
*that* for item `j`. `arr[i, j]` goes straight to the number in one step.

There's also something `arr[i][j]` simply cannot do: slice both directions at once.

In [6]:
print("arr2d[1][2] :", arr2d[1][2], "  ← works, but builds a throwaway row first")
print("arr2d[1, 2] :", arr2d[1, 2], "  ← goes straight there. Use this one.")

# Only the comma version can slice both directions together:
print("\nevery row, from column 1 onwards:\n", arr2d[:, 1:])

arr2d[1][2] : 6   ← works, but builds a throwaway row first
arr2d[1, 2] : 6   ← goes straight there. Use this one.

every row, from column 1 onwards:
 [[2 3]
 [5 6]
 [8 9]]


## 3. Filtering — "only give me the ones where..."

This is the single most useful thing in this notebook.

Compare an array to something and you don't get one `True`/`False` — you get **a whole array
of `True`/`False`, one per item**. That's called a **mask**, and you can think of it as a
stencil laid over your data: wherever the stencil says `True`, the value shows through.

Feed the mask back in as an index and you keep only the `True` positions. No loop, no `if`.

In [7]:
# ==========================================
# 3. FILTERING WITH A MASK
# ==========================================
numbers = np.array([1, 2, 3, 4, 5, 6, 7, 8])

# Step A: build the stencil. "Is each number even?"
mask = numbers % 2 == 0

# Step B: lay the stencil over the data and keep what shows through
evens = numbers[mask]

print("numbers      :", numbers)
print("mask         :", mask, "  ← one True/False per number")
print("numbers[mask]:", evens,   "  ← only the True positions survived")

# In real code you'd skip the middleman and write it in one line:
print("\neven numbers :", numbers[numbers % 2 == 0])
print("big numbers  :", numbers[numbers > 5])

numbers      : [1 2 3 4 5 6 7 8]
mask         : [False  True False  True False  True False  True]   ← one True/False per number
numbers[mask]: [2 4 6 8]   ← only the True positions survived

even numbers : [2 4 6 8]
big numbers  : [6 7 8]


### Combining conditions: use `&`, `|`, `~` — **never** `and`, `or`, `not`

This one catches everybody, so here's the reason.

`and` expects a single yes/no answer on each side. You're handing it eight answers at once,
so it doesn't know what to do and gives up with an error.

The symbol versions work item-by-item, which is what you actually want:

| You mean | Type | Not |
|---|---|---|
| both conditions | `&` | `and` |
| either condition | `\|` | `or` |
| flip it around | `~` | `not` |

⚠️ **Wrap each condition in its own brackets.** `&` grabs its neighbours more tightly than
`>` does, so without brackets Python groups the pieces wrongly.

In [8]:
# ==========================================
# 3b. COMBINING CONDITIONS
# ==========================================
# Note the brackets around EACH condition — they're not optional
print("bigger than 3 AND smaller than 7 :", numbers[(numbers > 3) & (numbers < 7)])
print("smaller than 3 OR bigger than 6  :", numbers[(numbers < 3) | (numbers > 6)])
print("NOT even                         :", numbers[~(numbers % 2 == 0)])

# Here's what happens with the English words instead:
try:
    numbers[(numbers > 3) and (numbers < 7)]
except ValueError as e:
    print("\nusing 'and' fails →", e)
    print("(It's asking: is a list of 8 answers True or False? There's no sensible reply.)")

bigger than 3 AND smaller than 7 : [4 5 6]
smaller than 3 OR bigger than 6  : [1 2 7 8]
NOT even                         : [1 3 5 7]

using 'and' fails → The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
(It's asking: is a list of 8 answers True or False? There's no sensible reply.)


### Asking questions about a mask

A mask is full of `True`/`False`, and Python counts `True` as 1 and `False` as 0. So:

- **`.sum()`** adds up the `True`s → **how many matched**
- **`.mean()`** is the average → **what fraction matched**, i.e. a percentage
- **`.any()`** → "was there at least one?"
- **`.all()`** → "were they all?"

And masks work on the left of an `=` too, which is how you fix up a batch of values at once.

In [9]:
# ==========================================
# 3c. COUNTING AND EDITING WITH MASKS
# ==========================================
scores = np.array([45, 88, 72, 91, 60, 35, 79])
passing = scores >= 60                # the stencil: who passed?

print("scores            :", scores)
print("how many passed   :", passing.sum(),        "← True counts as 1")
print("pass rate         :", f"{passing.mean():.1%}", "← the average of the True/Falses")
print("did anyone fail   :", (~passing).any(),     "← flip it, then ask 'any?'")
print("did everyone pass :", passing.all())
print("positions of passes:", np.flatnonzero(passing), "← which slots were True")

# Grading on a curve: bump every failing score up to 60, in one line
curved = scores.copy()                # copy first, so the originals stay intact
curved[curved < 60] = 60              # find the low ones and overwrite them
print("\noriginal scores   :", scores)
print("curved scores     :", curved)

scores            : [45 88 72 91 60 35 79]
how many passed   : 5 ← True counts as 1
pass rate         : 71.4% ← the average of the True/Falses
did anyone fail   : True ← flip it, then ask 'any?'
did everyone pass : False
positions of passes: [1 2 3 4 6] ← which slots were True

original scores   : [45 88 72 91 60 35 79]
curved scores     : [60 88 72 91 60 60 79]


## 4. Making decisions without writing `if`

You can't put a normal `if` statement in front of a whole array — an `if` needs one yes/no
answer, and an array has thousands. NumPy gives you array-shaped replacements:

| What you're trying to do | Use |
|---|---|
| "Tell me *where* the condition is true" | `np.where(condition)` → positions |
| "Use this value if true, that one if false" | `np.where(condition, if_true, if_false)` |
| "Several bands: A / B / C / fail" | `np.select([...], [...], default)` |
| "Squash everything into a sensible range" | `np.clip(arr, low, high)` |

`np.where` with three arguments is the important one. It's the array version of "if this
then that, otherwise the other" — applied to every item at once.

In [10]:
# ==========================================
# 4a. np.where — TWO DIFFERENT JOBS
# ==========================================
numbers = np.array([1, 2, 3, 4, 5])

# JOB 1: with one argument, it tells you WHERE (which positions)
positions = np.where(numbers > 3)
print("np.where(numbers > 3) :", positions, "← positions 3 and 4")
print("using those positions :", numbers[positions])
print("(the odd-looking comma is because it works on multi-dimensional arrays too)")

# JOB 2: with three arguments, it CHOOSES a value for every item
# Read as: where the number is over 2, use number*4; everywhere else, keep the number
print("\nquadruple anything over 2:", np.where(numbers > 2, numbers * 4, numbers))

np.where(numbers > 3) : (array([3, 4]),) ← positions 3 and 4
using those positions : [4 5]
(the odd-looking comma is because it works on multi-dimensional arrays too)

quadruple anything over 2: [ 1  2 12 16 20]


In [11]:
# ==========================================
# 4b. np.select AND np.clip
# ==========================================
scores = np.array([95, 82, 71, 65, 45])

# np.select handles several bands at once — like a chain of if/elif/else.
# It checks the conditions IN ORDER and uses the first one that matches.
grades = np.select(
    [scores >= 90, scores >= 80, scores >= 70, scores >= 60],   # the conditions
    ['A',          'B',          'C',          'D'],            # what to use for each
    default='F')                                                # if nothing matched
print("scores :", scores)
print("grades :", grades)

# np.clip forces everything inside a range: too low becomes the floor,
# too high becomes the ceiling. Useful for cleaning up bad readings.
raw = np.array([-5, 3, 12, 150, 60])
print("\nraw percentages     :", raw)
print("clipped to 0..100   :", np.clip(raw, 0, 100), "← -5 became 0, 150 became 100")

scores : [95 82 71 65 45]
grades : ['A' 'B' 'C' 'D' 'F']

raw percentages     : [ -5   3  12 150  60]
clipped to 0..100   : [  0   3  12 100  60] ← -5 became 0, 150 became 100


## 5. Sorting

Two versions, and the difference matters:
- **`np.sort(arr)`** hands back a sorted **copy**. Your original is untouched.
- **`arr.sort()`** sorts the original **in place** and returns nothing.

There's no `reverse=True` option. To sort largest-first, sort normally and then flip it with
`[::-1]`.

On a table, `axis` picks the direction — and **each row (or column) is sorted on its own**,
which is probably not what you want. More on that in a moment.

In [12]:
# ==========================================
# 5a. SORTING A SIMPLE ROW
# ==========================================
unsorted = np.array([3, 1, 4, 2, 5])

print("original        :", unsorted)
print("np.sort(...)    :", np.sort(unsorted))
print("original after  :", unsorted, "← untouched, np.sort gave back a copy")
print("largest first   :", np.sort(unsorted)[::-1], "← sort, then reverse")

original        : [3 1 4 2 5]
np.sort(...)    : [1 2 3 4 5]
original after  : [3 1 4 2 5] ← untouched, np.sort gave back a copy
largest first   : [5 4 3 2 1] ← sort, then reverse


In [13]:
# ==========================================
# 5b. SORTING A TABLE — WATCH THE ROWS BREAK APART
# ==========================================
arr2d = np.array([[3, 1, 4],
                  [2, 5, 6],
                  [9, 7, 8]])
print("original:\n", arr2d)

# axis=0 sorts DOWNWARDS, within each column separately
print("\naxis=0 — each column sorted top to bottom:\n", np.sort(arr2d, axis=0))

# axis=1 sorts ACROSS, within each row separately
print("\naxis=1 — each row sorted left to right:\n", np.sort(arr2d, axis=1))

print("\n⚠️  Notice the rows got shuffled apart. If these were [id, score] pairs,")
print("    the ids would no longer match their scores. The next cell fixes that.")

original:
 [[3 1 4]
 [2 5 6]
 [9 7 8]]

axis=0 — each column sorted top to bottom:
 [[2 1 4]
 [3 5 6]
 [9 7 8]]

axis=1 — each row sorted left to right:
 [[1 3 4]
 [2 5 6]
 [7 8 9]]

⚠️  Notice the rows got shuffled apart. If these were [id, score] pairs,
    the ids would no longer match their scores. The next cell fixes that.


### `argsort` — sorting a table without breaking it apart

This is how you sort a real table, the way clicking a spreadsheet column header does it:
**the whole row travels with the value you sorted on.**

`np.argsort` doesn't return sorted values. It returns **the positions those values would move
to** — a set of instructions like *"take item 1 first, then item 2, then item 0."*

Then you apply those instructions to the whole table, so every row moves as one piece.

In [14]:
# ==========================================
# 5c. argsort — SORT A TABLE BY ONE COLUMN
# ==========================================
values = np.array([30, 10, 20])
order  = np.argsort(values)

print("values  :", values)
print("argsort :", order, "← instructions: 'item 1 first, then item 2, then item 0'")
print("applied :", values[order])

# The real use: a table of [student_id, score], sorted by score, best first
table = np.array([[101, 72],
                  [102, 95],
                  [103, 60],
                  [104, 88]])

# Step A: get the score column               → [72, 95, 60, 88]
# Step B: work out the sorting instructions  → argsort
# Step C: reverse them for highest-first     → [::-1]
# Step D: apply them to the WHOLE table, so ids stay glued to their scores
ranked = table[np.argsort(table[:, 1])[::-1]]

print("\nsorted by score, best first:\n", ranked)
print("\ntop scorer's id:", ranked[0, 0], "← still correctly attached to the 95")

values  : [30 10 20]
argsort : [1 2 0] ← instructions: 'item 1 first, then item 2, then item 0'
applied : [10 20 30]

sorted by score, best first:
 [[102  95]
 [104  88]
 [101  72]
 [103  60]]

top scorer's id: 102 ← still correctly attached to the 95


## 6. Joining arrays together

Three names, one idea, and the difference is just *which direction*:

- **`np.vstack`** — **v**ertical: pile arrays on top of each other, adding **rows**
- **`np.hstack`** — **h**orizontal: set them side by side, adding **columns**
- **`np.concatenate`** — the general version; you tell it the direction with `axis`

There's also **`np.stack`**, which is different in kind: instead of merging arrays into one
bigger array, it keeps them separate and creates a whole new dimension to hold them —
like putting two sheets of paper in a folder rather than taping them together.

In [15]:
# ==========================================
# 6a. JOINING vs STACKING
# ==========================================
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

# concatenate: tape them end to end → still one flat row, now 6 long
joined = np.concatenate((a, b))
print("concatenate :", joined, "  shape", joined.shape, "← one longer row")

# stack: keep them as separate rows → a new 2-row table
stacked = np.stack((a, b))
print("stack       :\n", stacked, "  shape", stacked.shape, "← a new dimension appeared")

concatenate : [1 2 3 4 5 6]   shape (6,) ← one longer row
stack       :
 [[1 2 3]
 [4 5 6]]   shape (2, 3) ← a new dimension appeared


In [16]:
# ==========================================
# 6b. ADDING ROWS AND COLUMNS TO A TABLE
# ==========================================
original = np.array([[1, 2],
                     [3, 4]])

# vstack = add a row underneath. The new piece needs the same WIDTH.
new_row = np.array([[5, 6]])            # 1 row, 2 wide — matches
print("vstack, adding a row underneath:\n", np.vstack((original, new_row)))

# hstack = add a column on the side. The new piece needs the same HEIGHT.
new_col = np.array([[7], [8]])          # 2 rows, 1 wide — matches
print("\nhstack, adding a column on the right:\n", np.hstack((original, new_col)))

# column_stack takes plain rows and stands them up as columns.
# This is the easiest way to build a table out of separate lists.
ids    = np.array([101, 102, 103])
scores = np.array([72, 95, 60])
print("\ncolumn_stack turns two lists into a 2-column table:\n",
      np.column_stack((ids, scores)))

vstack, adding a row underneath:
 [[1 2]
 [3 4]
 [5 6]]

hstack, adding a column on the right:
 [[1 2 7]
 [3 4 8]]

column_stack turns two lists into a 2-column table:
 [[101  72]
 [102  95]
 [103  60]]


### The shape rule everybody trips on

To join two arrays, they have to match on **every direction except the one you're joining
along**. Think of taping two sheets of paper together — the edges you're taping have to be
the same length.

- Stacking **rows** (`vstack`) → the arrays need the same **width**
- Stacking **columns** (`hstack`) → the arrays need the same **height**

In [17]:
a    = np.array([[1, 2], [3, 4]])       # 2 rows, 2 wide
good = np.array([[5, 6]])               # 1 row,  2 wide → widths match, this will work
bad  = np.array([[5, 6, 7]])            # 1 row,  3 wide → too wide, this will not

print("a is    ", a.shape)
print("good is ", good.shape, "→ same width?", a.shape[1] == good.shape[1])
print("bad is  ", bad.shape,  "→ same width?", a.shape[1] == bad.shape[1])

try:
    np.vstack((a, bad))
except ValueError as e:
    print("\nAs expected:", e)

a is     (2, 2)
good is  (1, 2) → same width? True
bad is   (1, 3) → same width? False

As expected: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 1, the array at index 0 has size 2 and the array at index 1 has size 3


## 7. Adding and removing items

⚠️ **Arrays are a fixed size.** Unlike a Python list, you cannot really add to one.

`np.append`, `np.insert` and `np.delete` *look* like they modify your array, but they don't
touch it. Each one **builds a brand-new array and copies everything across.** They always
return the result — if you don't catch it in a variable, nothing happens.

### Why this matters: never call `np.append` in a loop

Each call copies the entire array. Adding 10,000 items one at a time means 10,000 copies of
a steadily growing array — that's the difference between finishing instantly and waiting
minutes.

**Do this instead:** collect into a normal Python list (which *is* built for growing) and
convert once at the end. Or, if you know the final size, make the array up front with
`np.zeros` and fill in the slots.

In [18]:
# ==========================================
# 7a. THESE ALL RETURN A NEW ARRAY
# ==========================================
arr = np.array([1, 2, 3, 4, 5])

print("original            :", arr)
print("delete position 2   :", np.delete(arr, 2))
print("delete 0 and 4      :", np.delete(arr, [0, 4]))
print("insert 99 at pos 2  :", np.insert(arr, 2, 99))
print("append 6 on the end :", np.append(arr, 6))
print("\noriginal, still    :", arr, "← never changed. Each line above made a fresh array.")

original            : [1 2 3 4 5]
delete position 2   : [1 2 4 5]
delete 0 and 4      : [2 3 4]
insert 99 at pos 2  : [ 1  2 99  3  4  5]
append 6 on the end : [1 2 3 4 5 6]

original, still    : [1 2 3 4 5] ← never changed. Each line above made a fresh array.


In [19]:
# ==========================================
# 7b. THE RIGHT WAY TO BUILD AN ARRAY PIECE BY PIECE
# ==========================================

# OPTION 1: use a Python list while collecting, convert once at the end.
# Lists are designed to grow; arrays are not.
squares = [x ** 2 for x in range(10)]      # collect...
squares = np.array(squares)                # ...and convert, one single time
print("built via a list :", squares)

# OPTION 2: if you already know how many you need, make the array first and fill it in.
out = np.zeros(10, dtype=int)              # 10 empty slots, ready to go
for i in range(10):
    out[i] = i ** 2                        # drop each answer into its slot — no copying
print("pre-made and filled:", out)

built via a list : [ 0  1  4  9 16 25 36 49 64 81]
pre-made and filled: [ 0  1  4  9 16 25 36 49 64 81]


## 8. Cutting an array into pieces

The opposite of stacking. `np.split` insists the pieces come out equal and complains if they
can't; `np.array_split` is relaxed about it and makes them as even as possible.

In [20]:
# ==========================================
# 8. SPLITTING
# ==========================================
arr = np.arange(9)                          # 0 to 8 — nine items

print("split into 3 equal parts:", np.split(arr, 3))
print("array_split into 4 parts:", np.array_split(arr, 4), "← 9 doesn't divide by 4, that's fine")

# On a table: hsplit cuts vertically (left/right), vsplit cuts horizontally (top/bottom)
grid = np.arange(12).reshape(3, 4)
print("\nthe grid:\n", grid)
left, right = np.hsplit(grid, 2)            # slice it down the middle
print("\nleft half:\n", left, "\n\nright half:\n", right)

split into 3 equal parts: [array([0, 1, 2]), array([3, 4, 5]), array([6, 7, 8])]
array_split into 4 parts: [array([0, 1, 2]), array([3, 4]), array([5, 6]), array([7, 8])] ← 9 doesn't divide by 4, that's fine

the grid:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

left half:
 [[0 1]
 [4 5]
 [8 9]] 

right half:
 [[ 2  3]
 [ 6  7]
 [10 11]]


## Cheat sheet

| What you want | What to type |
|---|---|
| A range of items | `a[start:stop:step]` — and `a[::-1]` to reverse |
| One row / one column | `a[i, :]` / `a[:, j]` |
| Only the items matching a rule | `a[a > 5]`, joined with `&` `\|` `~` and brackets |
| How many matched | `(a > 5).sum()`, `.any()`, `.all()` |
| Choose per item | `np.where(cond, yes, no)`, `np.select`, `np.clip` |
| Sort | `np.sort(a, axis=...)` |
| Sort a table by a column | `table[np.argsort(table[:, col])]` |
| Join | `np.vstack`, `np.hstack`, `np.concatenate`, `np.column_stack` |
| Add / remove | `np.insert`, `np.delete`, `np.append` — all return a **new** array |
| Cut up | `np.split`, `np.array_split`, `np.hsplit`, `np.vsplit` |

### The three things worth memorising

1. **`&` `|` `~` with brackets around each condition.** The words `and`/`or` will error.
2. **Slices are windows; filters and list-indexing give copies.** (Phase 1's lesson again.)
3. **Never `np.append` inside a loop.** Build a Python list, convert once at the end.

### Practice

1. From `np.arange(1, 21)`, pull out the numbers that are divisible by 3 **and** bigger
   than 10.
2. Sort `np.array([[5, 2], [1, 9], [3, 7]])` by its **second** column, keeping rows intact.
3. Turn every negative number in `np.array([-3, 5, -1, 8])` into 0 — do it twice, once with
   `np.where` and once with a mask.
4. Stack `np.arange(3)` and `np.arange(3, 6)` into a 2-row table, then into a 3-row one.

In [21]:
# ==========================================
# PRACTICE 1 — worked solution
# ==========================================
a = np.arange(1, 21)
print(a[(a % 3 == 0) & (a > 10)])       # both conditions, each in its own brackets

# ==========================================
# PRACTICE 2 — worked solution
# ==========================================
t = np.array([[5, 2], [1, 9], [3, 7]])
print(t[np.argsort(t[:, 1])])           # t[:, 1] is the second column; sort the table by it

[12 15 18]
[[5 2]
 [3 7]
 [1 9]]
